# NHANES 2021-2023 - Análisis Pre-ML

## Descripción del Dataset
NHANES (National Health and Nutrition Examination Survey) es una encuesta continua del CDC (Centers for Disease Control) que evalúa la salud y nutrición de adultos y niños en Estados Unidos mediante entrevistas, exámenes físicos y pruebas de laboratorio.

**Dataset utilizado:** DEMO_L.xpt (Demographic Variables and Sample Weights)

**Variables principales:**
- `SEQN`: ID único del participante
- `RIAGENDR`: Género (1=Masculino, 2=Femenino)
- `RIDAGEYR`: Edad en años
- `RIDRETH3`: Raza/Etnia
- `DMDEDUC2`: Nivel educativo
- `DMDMARTL`: Estado civil
- `INDFMPIR`: Índice de pobreza familiar (ratio ingreso/umbral de pobreza)

## Objetivo de Machine Learning
**Regresión**: Predecir el nivel socioeconómico de un participante (`INDFMPIR`)

**Target**: `INDFMPIR` (Family Poverty Income Ratio)
- Valores < 1.0 = por debajo del umbral de pobreza
- Valores = 1.0-2.0 = cerca del umbral
- Valores > 2.0 = por encima del umbral de pobreza

**Features potenciales**:
- Demográficas: edad, género, raza/etnia
- Sociales: educación, estado civil, tamaño del hogar
- Geográficas: país de nacimiento

## PASO 1: Cargar Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Cargar archivo .xpt (formato SAS)
df = pd.read_sas('DEMO_J.xpt', format='xport', encoding='utf-8')

print("✅ Dataset cargado correctamente")

## PASO 2: Ver Estructura del Dataset

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes.to_frame(name='dtype')

In [ ]:
df.info()

## PASO 3: Valores Faltantes

In [ ]:
# Contar valores faltantes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print("=== VALORES FALTANTES ===")
if len(missing_df) > 0:
    print(missing_df.head(15))
else:
    print("✅ No hay valores faltantes")

## PASO 4: Gráfico del Target

In [ ]:
# Distribución del target (eliminar NaN para visualización)
target_clean = df['INDFMPIR'].dropna()

plt.figure(figsize=(10, 5))
plt.hist(target_clean, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
plt.title('Distribución del Índice de Pobreza Familiar (INDFMPIR)', fontsize=14, fontweight='bold')
plt.xlabel('INDFMPIR (ratio ingreso/umbral pobreza)')
plt.ylabel('Frecuencia')
plt.axvline(x=1.0, color='red', linestyle='--', label='Umbral de pobreza')
plt.axvline(x=2.0, color='orange', linestyle='--', label='2x umbral')
plt.legend()
plt.tight_layout()
plt.show()

# Estadísticas del target
print("\n=== ESTADÍSTICAS DEL TARGET ===")
print(target_clean.describe())
print(f"\nCasos por debajo del umbral de pobreza (<1.0): {(target_clean < 1.0).sum()} ({(target_clean < 1.0).sum()/len(target_clean)*100:.1f}%)")
print(f"Casos en el umbral (1.0-2.0): {((target_clean >= 1.0) & (target_clean < 2.0)).sum()} ({((target_clean >= 1.0) & (target_clean < 2.0)).sum()/len(target_clean)*100:.1f}%)")
print(f"Casos por encima (>=2.0): {(target_clean >= 2.0).sum()} ({(target_clean >= 2.0).sum()/len(target_clean)*100:.1f}%)")

---
# LIMPIEZA DE DATOS

## Seleccionar Variables Relevantes

In [ ]:
# Seleccionar columnas relevantes para el análisis
columns_to_keep = [
    'INDFMPIR',      # TARGET: Índice de pobreza familiar
    'RIAGENDR',      # Género
    'RIDAGEYR',      # Edad en años
    'RIDRETH3',      # Raza/Etnia
    'DMDEDUC2',      # Nivel educativo (adultos)
    'DMDMARTL',      # Estado civil
    'DMDHHSIZ',      # Tamaño del hogar
    'DMDFMSIZ',      # Tamaño de la familia
    'DMDBORN4',      # País de nacimiento
    'DMDCITZN'       # Ciudadanía
]

# Filtrar solo columnas que existen en el dataset
columns_available = [col for col in columns_to_keep if col in df.columns]
df_clean = df[columns_available].copy()

print(f"Columnas seleccionadas: {len(columns_available)}")
print(f"Shape inicial: {df_clean.shape}")

## Tratamiento de Valores Faltantes

In [ ]:
# Eliminar filas donde el target está ausente (no se puede predecir sin target)
print(f"Filas antes de eliminar target NaN: {len(df_clean)}")
df_clean = df_clean.dropna(subset=['INDFMPIR'])
print(f"Filas después de eliminar target NaN: {len(df_clean)}")

# Para el resto de variables: imputar con mediana (numéricas) o moda (categóricas)
for col in df_clean.columns:
    if col == 'INDFMPIR':  # Ya limpiamos el target
        continue
    
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['float64', 'int64']:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        else:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Verificar
remaining_nulls = df_clean.isnull().sum().sum()
print(f"\nValores faltantes después de limpieza: {remaining_nulls}")

if remaining_nulls == 0:
    print("✅ Todos los valores faltantes fueron tratados")
else:
    print(f"⚠️ Quedan {remaining_nulls} valores faltantes")

## Tratamiento de Outliers

In [ ]:
# Estadísticas de variables numéricas
print("=== ESTADÍSTICAS NUMÉRICAS ===")
print(df_clean.describe())

# En NHANES, los valores extremos de INDFMPIR están truncados a 5.0
print(f"\nValores de INDFMPIR = 5.0 (truncados): {(df_clean['INDFMPIR'] == 5.0).sum()}")
print("Nota: CDC trunca INDFMPIR en 5.0 para proteger privacidad de personas con ingresos muy altos")

print(f"\n✅ No se requiere tratamiento de outliers (los valores extremos son por diseño del dataset)")

---
# PREPROCESAMIENTO

## Encoding de Variables Categóricas

In [ ]:
df_encoded = df_clean.copy()

# Todas las variables son numéricas codificadas (NHANES usa códigos numéricos)
# No necesitamos one-hot encoding, pero verificamos tipos de datos

print("=== TIPOS DE DATOS ===")
print(df_encoded.dtypes)

# Asegurar que todas las columnas sean numéricas
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        # Convertir a numérico si es posible
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')

print(f"\n✅ Variables ya están codificadas numéricamente")
print(f"Shape: {df_encoded.shape}")

## Separar Features y Target

In [ ]:
# Target
y = df_encoded['INDFMPIR'].values

# Features (todas excepto el target)
X = df_encoded.drop('INDFMPIR', axis=1).values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nNombre de features:")
print(df_encoded.drop('INDFMPIR', axis=1).columns.tolist())

## Train-Test Split Manual (80-20)

In [ ]:
# Configuración
np.random.seed(42)
test_size = 0.2

# Generar índices aleatorios
n_samples = len(X)
indices = np.random.permutation(n_samples)

# Calcular punto de corte
test_size_abs = int(n_samples * test_size)
train_size_abs = n_samples - test_size_abs

# Dividir índices
train_indices = indices[:train_size_abs]
test_indices = indices[train_size_abs:]

# Crear conjuntos
X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print(f"✅ Split completado")
print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")
print(f"\nINDFMPIR promedio en Train: {y_train.mean():.2f}")
print(f"INDFMPIR promedio en Test: {y_test.mean():.2f}")

## Escalado Manual (StandardScaler)

Fórmula: `x_scaled = (x - mean) / std`

**IMPORTANTE**: Calcular media y desviación estándar SOLO con datos de entrenamiento para evitar data leakage.

In [ ]:
# Calcular media y desviación estándar del TRAIN set
mean_train = X_train.mean(axis=0)
std_train = X_train.std(axis=0)

# Evitar división por cero (columnas constantes)
std_train[std_train == 0] = 1

# Aplicar escalado
X_train_scaled = (X_train - mean_train) / std_train
X_test_scaled = (X_test - mean_train) / std_train  # Usar estadísticas del train

# Convertir a float explícitamente (prevenir errores de tipo)
X_train_scaled = X_train_scaled.astype(float)
X_test_scaled = X_test_scaled.astype(float)

print("✅ Escalado completado")
print(f"\nMedia de X_train_scaled (primeras 5 features): {X_train_scaled.mean(axis=0)[:5]}")
print(f"Desv. Std de X_train_scaled (primeras 5 features): {X_train_scaled.std(axis=0)[:5]}")

---
# VALIDACIÓN FINAL

## 1. Verificar Data Leakage

In [ ]:
# Verificar que no hay índices compartidos entre train y test
shared_indices = np.intersect1d(train_indices, test_indices)

if len(shared_indices) == 0:
    print("✅ No hay data leakage (índices únicos en train y test)")
else:
    print(f"⚠️ ALERTA: {len(shared_indices)} índices compartidos detectados")

## 2. Verificar NaN e Infinitos

In [ ]:
# Verificar NaN
nan_train = np.isnan(X_train_scaled).sum()
nan_test = np.isnan(X_test_scaled).sum()

# Verificar infinitos
inf_train = np.isinf(X_train_scaled).sum()
inf_test = np.isinf(X_test_scaled).sum()

print(f"NaN en X_train_scaled: {nan_train}")
print(f"NaN en X_test_scaled: {nan_test}")
print(f"Infinitos en X_train_scaled: {inf_train}")
print(f"Infinitos en X_test_scaled: {inf_test}")

if nan_train == 0 and nan_test == 0 and inf_train == 0 and inf_test == 0:
    print("\n✅ Datos limpios (sin NaN ni infinitos)")
else:
    print("\n⚠️ ALERTA: Hay NaN o infinitos en los datos")

## 3. Comparar Distribuciones Train vs Test

In [ ]:
# Comparar media y std de todas las features
print("=== COMPARACIÓN TRAIN vs TEST (todas las features) ===")
print("\nMEDIA:")
print(f"Train: {X_train_scaled.mean(axis=0)}")
print(f"Test:  {X_test_scaled.mean(axis=0)}")

print("\nDESVIACIÓN ESTÁNDAR:")
print(f"Train: {X_train_scaled.std(axis=0)}")
print(f"Test:  {X_test_scaled.std(axis=0)}")

print("\n✅ Las distribuciones deberían ser similares (train estandarizado, test escalado con stats de train)")

## Resumen Final

In [ ]:
print("="*60)
print("RESUMEN DEL PREPROCESAMIENTO")
print("="*60)
print(f"Dataset original: {df.shape}")
print(f"Dataset después de limpieza: {df_clean.shape}")
print(f"Features seleccionadas: {X.shape[1]}")
print(f"Train set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"\nTarget (INDFMPIR - Índice de Pobreza Familiar):")
print(f"  - Mínimo: {y.min():.2f}")
print(f"  - Promedio: {y.mean():.2f}")
print(f"  - Máximo: {y.max():.2f}")
print(f"  - Por debajo del umbral de pobreza (<1.0): {(y < 1.0).sum()} casos")
print(f"\n✅ Datos listos para entrenamiento de modelos de regresión")
print("="*60)